In [1]:
import sys
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

from src.config import TRAIN_PATH, TEST_PATH, RANDOM_SEED, CV_N_SPLITS
from src.preprocessing import HousePricesPreprocessor

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [3]:
y = np.log1p(train['SalePrice'])
test_ids = test['Id']

# Выбросы
outlier_idx = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000)].index
train = train.drop(outlier_idx)
y = y.drop(outlier_idx)

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing
preprocessor = HousePricesPreprocessor(scale=True)
X = preprocessor.fit_transform(train, np.expm1(y))
X_test = preprocessor.transform(test)

print(f"X: {X.shape}, y: {y.shape}")

X: (1458, 99), y: (1458,)


In [4]:
def train_dnn(model_fn, X, y, n_splits=CV_N_SPLITS, epochs=100, lr=0.001, batch_size=64):
    """KFold валидация для PyTorch моделей."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train = torch.FloatTensor(X.iloc[train_idx].values)
        y_train = torch.FloatTensor(y.iloc[train_idx].values).unsqueeze(1)
        X_val = torch.FloatTensor(X.iloc[val_idx].values)
        y_val = torch.FloatTensor(y.iloc[val_idx].values).unsqueeze(1)

        train_ds = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        model = model_fn()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        # Обучение
        model.train()
        for epoch in range(epochs):
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                pred = model(X_batch)
                loss = loss_fn(pred, y_batch)
                loss.backward()
                optimizer.step()

        # Валидация
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            rmse = torch.sqrt(loss_fn(val_pred, y_val)).item()
            scores.append(rmse)

    scores = np.array(scores)
    print(f"RMSE по фолдам: {scores.round(4)}")
    print(f"Среднее: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores

In [5]:
input_size = X.shape[1]  # 99

def simple_dnn():
    return nn.Sequential(
        nn.Linear(input_size, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 1)
    )

Simple DNN

In [6]:
scores = train_dnn(simple_dnn, X, y, epochs=100, lr=0.001)

RMSE по фолдам: [0.7557 0.8158 0.7    0.7835 1.0287]
Среднее: 0.8167 ± 0.1126


DNN: тюнинг lr и epochs

In [7]:
for lr in [0.01, 0.001, 0.0005, 0.0001]:
    for epochs in [200, 500]:
        print(f"\nlr={lr}, epochs={epochs}:")
        train_dnn(simple_dnn, X, y, epochs=epochs, lr=lr)


lr=0.01, epochs=200:
RMSE по фолдам: [0.1509 0.2021 0.2113 0.2017 0.17  ]
Среднее: 0.1872 ± 0.0229

lr=0.01, epochs=500:
RMSE по фолдам: [0.1214 0.154  0.1405 0.1561 0.3873]
Среднее: 0.1919 ± 0.0985

lr=0.001, epochs=200:
RMSE по фолдам: [0.5879 0.4505 0.502  0.4526 0.7085]
Среднее: 0.5403 ± 0.0978

lr=0.001, epochs=500:
RMSE по фолдам: [0.2482 0.2283 0.2638 0.1671 0.2458]
Среднее: 0.2306 ± 0.0337

lr=0.0005, epochs=200:
RMSE по фолдам: [0.5883 0.7072 0.5992 0.7415 1.0693]
Среднее: 0.7411 ± 0.1746

lr=0.0005, epochs=500:
RMSE по фолдам: [0.1649 0.3024 0.2644 0.3048 0.3073]
Среднее: 0.2688 ± 0.0542

lr=0.0001, epochs=200:
RMSE по фолдам: [1.0157 1.0008 0.9203 0.9692 1.1605]
Среднее: 1.0133 ± 0.0805

lr=0.0001, epochs=500:
RMSE по фолдам: [0.7413 0.7985 0.6627 0.8187 1.0783]
Среднее: 0.8199 ± 0.1401


In [8]:
def better_dnn():
    return nn.Sequential(
        nn.Linear(input_size, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

Better DNN (lr=0.01)

In [9]:
for epochs in [500, 1000]:
    print(f"\nepochs={epochs}:")
    train_dnn(better_dnn, X, y, epochs=epochs, lr=0.01)


epochs=500:
RMSE по фолдам: [0.1255 0.1382 0.118  0.1399 0.1442]
Среднее: 0.1331 ± 0.0098

epochs=1000:
RMSE по фолдам: [0.1395 0.149  0.1192 0.1483 0.2031]
Среднее: 0.1518 ± 0.0278


DNN: итоги
- Simple DNN (lr=0.01, epochs=500):    CV=0.1649
- Better DNN (BatchNorm, epochs=1000): CV=0.1441